# Modelado de Churn

## Proyecto: SaaS Customer Churn Intelligence

El objetivo de este notebook consiste en construir un modelo que permita predecir el abandono de un cliente. 


## 1. Carga de los datos

Cargamos el archivo `features.parquet`, que contiene las variables preparadas en la fase de Feature Engineering.


In [1]:
import pandas as pd
import numpy as np

df = pd.read_parquet("../data/processed/features.parquet")

print("Shape:", df.shape)
display(df.head())


Shape: (7043, 29)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,TotalCharges,Churn,tenure_group,monthly_charge_group,is_monthly_contract,num_services_contracted,has_tech_support,has_online_security,uses_electronic_check,churn_target
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,29.85,No,0-12 months,Low,1,1,0,0,1,0
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,1889.50,No,25-48 months,Medium,0,2,0,1,0,0
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,108.15,Yes,0-12 months,Medium,1,2,0,1,0,1
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,1840.75,No,25-48 months,Low,0,3,1,1,0,0
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,151.65,Yes,0-12 months,Medium,1,0,0,0,1,1


## 2. Definición del target y de las variables predictoras

La variable que queremos predecir es `Churn`.

En Feature Engineering también se creó `churn_target`, pero no la utilizaremos como predictor porque representa la misma información que `Churn`.

`customerID` tampoco se utilizará para entrenar los modelos, ya que solo sirve para identificar al cliente.


In [2]:
target_column = "Churn"
id_column = "customerID"
target_copy_column = "churn_target"

required_columns = [target_column, id_column]

for column in required_columns:
    if column not in df.columns:
        raise ValueError(f"No se encuentra la columna requerida: {column}")

print("Target:", target_column)
print("Identificador:", id_column)
print("¿Existe churn_target?:", target_copy_column in df.columns)


Target: Churn
Identificador: customerID
¿Existe churn_target?: True


## 3. Revisión de variables originales y derivadas

En la fase anterior se crearon varias variables derivadas a partir de variables que ya existían. Antes de entrenar los modelos revisamos estas parejas para no incluir información duplicada sin necesidad.

| Variable original | Variable derivada | Decisión |
|---|---|---|
| `tenure` | `tenure_group` | Mantener las dos |
| `MonthlyCharges` | `monthly_charge_group` | Mantener las dos |
| `Contract` | `is_monthly_contract` | Mantener las dos |
| `TechSupport` | `has_tech_support` | Mantener solo la original |
| `OnlineSecurity` | `has_online_security` | Mantener solo la original |
| `PaymentMethod` | `uses_electronic_check` | Mantener las dos |
| Servicios adicionales | `num_services_contracted` | Mantener el contador |

Las variables `has_tech_support` y `has_online_security` se excluyen porque aportan prácticamente la misma información que sus variables originales. El resto se mantiene porque puede aportar una representación diferente y fácil de interpretar.


In [3]:
# Variables derivadas que se consideran redundantes con su variable original.
redundant_features = [
    "has_tech_support",
    "has_online_security"
]

# La variable objetivo y el identificador tampoco forman parte de X.
excluded_columns = [
    target_column,
    id_column,
    target_copy_column
]

# Excluimos las representaciones derivadas que duplican prácticamente
# la información de una variable categórica que ya conservamos.
excluded_columns.extend([c for c in redundant_features if c in df.columns])

feature_columns = [
    c for c in df.columns
    if c not in excluded_columns
]

X = df[feature_columns].copy()
y = df[target_column].copy()

print("Número de variables predictoras:", len(feature_columns))
print("Variables excluidas:", excluded_columns)
print("\nVariables utilizadas:")
print(feature_columns)


Número de variables predictoras: 24
Variables excluidas: ['Churn', 'customerID', 'churn_target', 'has_tech_support', 'has_online_security']

Variables utilizadas:
['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'tenure_group', 'monthly_charge_group', 'is_monthly_contract', 'num_services_contracted', 'uses_electronic_check']


## 4. División Train/Test

Antes de preparar los datos hacemos la división entre entrenamiento y prueba.

Utilizamos un 80% para train y un 20% para test. Además, usamos `stratify` para que la proporción de clientes con y sin churn sea muy parecida en ambos grupos.

Esto también nos permite calcular algunas transformaciones utilizando solo train y aplicar después esas mismas transformaciones a test.


In [4]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

print("Train:", X_train.shape)
print("Test:", X_test.shape)

print("\nDistribución de Churn en train:")
print(y_train.value_counts(normalize=True))

print("\nDistribución de Churn en test:")
print(y_test.value_counts(normalize=True))


Train: (5634, 24)
Test: (1409, 24)

Distribución de Churn en train:
Churn
No     0.734647
Yes    0.265353
Name: proportion, dtype: float64

Distribución de Churn en test:
Churn
No     0.734564
Yes    0.265436
Name: proportion, dtype: float64


## 5. Preparación de los datos para Machine Learning

Ahora dejamos los datos preparados para los modelos.

Primero tratamos los valores que faltan en `TotalCharges`. Después recalculamos `monthly_charge_group` usando solo el conjunto de entrenamiento. Así evitamos utilizar información del test para construir una variable que después usaremos en el modelo.

Por último, convertimos las variables categóricas a variables numéricas mediante One-Hot Encoding.


In [5]:
X_train = X_train.copy()
X_test = X_test.copy()

# Tratamiento sencillo del valor ausente de TotalCharges.
if "TotalCharges" in X_train.columns:
    X_train["TotalCharges"] = pd.to_numeric(
        X_train["TotalCharges"], errors="coerce"
    ).fillna(0)

if "TotalCharges" in X_test.columns:
    X_test["TotalCharges"] = pd.to_numeric(
        X_test["TotalCharges"], errors="coerce"
    ).fillna(0)

# Recalcular monthly_charge_group usando solamente train.
if "MonthlyCharges" in X_train.columns:
    monthly_charge_q1 = X_train["MonthlyCharges"].quantile(1/3)
    monthly_charge_q2 = X_train["MonthlyCharges"].quantile(2/3)

    bins = [-np.inf, monthly_charge_q1, monthly_charge_q2, np.inf]
    labels = ["low", "medium", "high"]

    X_train["monthly_charge_group"] = pd.cut(
        X_train["MonthlyCharges"],
        bins=bins,
        labels=labels
    )

    X_test["monthly_charge_group"] = pd.cut(
        X_test["MonthlyCharges"],
        bins=bins,
        labels=labels
    )

categorical_columns = X_train.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

X_train_encoded = pd.get_dummies(
    X_train,
    columns=categorical_columns,
    drop_first=True,
    dtype=int
)

X_test_encoded = pd.get_dummies(
    X_test,
    columns=categorical_columns,
    drop_first=True,
    dtype=int
)

# Aseguramos que test tenga exactamente las mismas columnas que train.
X_test_encoded = X_test_encoded.reindex(
    columns=X_train_encoded.columns,
    fill_value=0
)

print("Train después del encoding:", X_train_encoded.shape)
print("Test después del encoding:", X_test_encoded.shape)


Train después del encoding: (5634, 38)
Test después del encoding: (1409, 38)


C:\Users\alebo\AppData\Local\Temp\ipykernel_2016\3857718673.py:35: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_columns = X_train.select_dtypes(


### Comprobación de la preparación

Antes de entrenar comprobamos que la preparación se ha realizado correctamente.

El resultado muestra que no quedan valores nulos, que todas las variables son numéricas y que train y test tienen las mismas columnas.

Con estas comprobaciones podemos continuar con el entrenamiento.


In [6]:
print("Valores nulos en train:", X_train_encoded.isna().sum().sum())
print("Valores nulos en test:", X_test_encoded.isna().sum().sum())
print("¿Mismas columnas?:", list(X_train_encoded.columns) == list(X_test_encoded.columns))
print("¿Todas las columnas son numéricas?:",
      all(pd.api.types.is_numeric_dtype(dtype) for dtype in X_train_encoded.dtypes))


Valores nulos en train: 0
Valores nulos en test: 0
¿Mismas columnas?: True
¿Todas las columnas son numéricas?: True


## 6. Modelo baseline: Regresión Logística

Este tipo de modelo es apropiado para problemas como este de clasificación binaria. Además permite interpretar la dirección y el peso de las variables mediante sus coeficientes.

In [7]:
from sklearn.linear_model import LogisticRegression

logistic_model = LogisticRegression(
    max_iter=1000,
    solver="liblinear",
    random_state=42
)

logistic_model.fit(X_train_encoded, y_train)

print("Regresión Logística entrenada.")


Regresión Logística entrenada.


## 7. Evaluación del modelo baseline

Evaluamos la Regresión Logística utilizando:
- Accuracy
- Precision
- Recall
- F1-score
- matriz de confusión.

En este proyecto prestamos especial atención al Recall, porque un False Negative significa que el modelo no detecta a un cliente que realmente abandona.

También miraremos el resto de métricas para no basar la decisión en un único valor.


In [8]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

y_pred_logistic = logistic_model.predict(X_test_encoded)

logistic_metrics = {
    "Model": "Logistic Regression",
    "Accuracy": accuracy_score(y_test, y_pred_logistic),
    "Precision": precision_score(y_test, y_pred_logistic, pos_label="Yes", zero_division=0),
    "Recall": recall_score(y_test, y_pred_logistic, pos_label="Yes", zero_division=0),
    "F1": f1_score(y_test, y_pred_logistic, pos_label="Yes", zero_division=0)
}

print(logistic_metrics)
print("\nMatriz de confusión:")
print(confusion_matrix(y_test, y_pred_logistic, labels=["No", "Yes"]))


{'Model': 'Logistic Regression', 'Accuracy': 0.8005677785663591, 'Precision': 0.6495176848874598, 'Recall': 0.5401069518716578, 'F1': 0.5897810218978102}

Matriz de confusión:
[[926 109]
 [172 202]]


## 8. Segundo modelo: Decision Tree

Como segundo modelo utilizamos un Decision Tree.

La idea es comparar un modelo lineal con otro basado en reglas. Limitamos la profundidad a 5 para mantenerlo sencillo y fácil de interpretar.


In [9]:
from sklearn.tree import DecisionTreeClassifier

tree_model = DecisionTreeClassifier(
    max_depth=5,
    random_state=42
)

tree_model.fit(X_train_encoded, y_train)

print("Decision Tree entrenado.")


Decision Tree entrenado.


## 9. Evaluación del segundo modelo

Evaluamos el Decision Tree con las mismas métricas utilizadas para la Regresión Logística.

De esta forma podemos comparar los dos modelos en las mismas condiciones.


In [10]:
y_pred_tree = tree_model.predict(X_test_encoded)

tree_metrics = {
    "Model": "Decision Tree",
    "Accuracy": accuracy_score(y_test, y_pred_tree),
    "Precision": precision_score(y_test, y_pred_tree, pos_label="Yes", zero_division=0),
    "Recall": recall_score(y_test, y_pred_tree, pos_label="Yes", zero_division=0),
    "F1": f1_score(y_test, y_pred_tree, pos_label="Yes", zero_division=0)
}

print(tree_metrics)
print("\nMatriz de confusión:")
print(confusion_matrix(y_test, y_pred_tree, labels=["No", "Yes"]))


{'Model': 'Decision Tree', 'Accuracy': 0.7991483321504613, 'Precision': 0.6318840579710145, 'Recall': 0.5828877005347594, 'F1': 0.6063977746870653}

Matriz de confusión:
[[908 127]
 [156 218]]


## 10. Comparación de los modelos

Ahora podemos comparar los resultados de los dos modelos.

Nos fijaremos especialmente en Recall y F1, pero también en Accuracy y Precision. 

In [11]:
results = pd.DataFrame([logistic_metrics, tree_metrics]).set_index("Model")
display(results.round(3))


,Accuracy,Precision,Recall,F1
Model,,,,
Logistic Regression,0.801,0.650,0.540,0.590
Decision Tree,0.799,0.632,0.583,0.606


### Resultado observado

Los resultados obtenidos son los siguientes:

- La **Regresión Logística** obtiene una Accuracy de **0.801**, una Precision de **0.650**, un Recall de **0.540** y un F1 de **0.590**.
- El **Decision Tree** obtiene una Accuracy de **0.799**, una Precision de **0.632**, un Recall de **0.583** y un F1 de **0.606**.

Por tanto, el Decision Tree detecta una mayor proporción de clientes que hacen churn y también consigue un F1 algo mejor. La Regresión Logística, en cambio, obtiene una Accuracy y una Precision ligeramente superiores.

La diferencia entre los modelos se entenderá mejor al revisar los errores.


## 11. Interpretación de las variables

Una vez vistos los resultados, podemos revisar qué variables tienen más peso en cada modelo.

En la Regresión Logística utilizamos los coeficientes. En el Decision Tree utilizamos la importancia de las variables.

Estas medidas nos ayudan a entender el modelo, pero no significan que una variable sea la causa del churn.


In [12]:
# Coeficientes de la Regresión Logística
logistic_importance = pd.DataFrame({
    "Feature": X_train_encoded.columns,
    "Coefficient": logistic_model.coef_[0]
})

logistic_importance["Abs_Coefficient"] = logistic_importance["Coefficient"].abs()

print("Variables con mayor peso absoluto en la Regresión Logística:")
display(
    logistic_importance
    .sort_values("Abs_Coefficient", ascending=False)
    .head(15)
)

# Importancias del Decision Tree
tree_importance = pd.DataFrame({
    "Feature": X_train_encoded.columns,
    "Importance": tree_model.feature_importances_
})

print("Variables con mayor importancia en el Decision Tree:")
display(
    tree_importance
    .sort_values("Importance", ascending=False)
    .head(15)
)


Variables con mayor peso absoluto en la Regresión Logística:


,Feature,Coefficient,Abs_Coefficient
28,Contract_Two year,-0.535472,0.535472
13,InternetService_Fiber optic,0.480443,0.480443
10,PhoneService_Yes,-0.438999,0.438999
4,is_monthly_contract,0.422726,0.422726
29,PaperlessBilling_Yes,0.378260,0.378260
16,OnlineSecurity_Yes,-0.359209,0.359209
33,tenure_group_13-24 months,-0.321726,0.321726
22,TechSupport_Yes,-0.321164,0.321164
24,StreamingTV_Yes,0.271944,0.271944
26,StreamingMovies_Yes,0.269713,0.269713


Variables con mayor importancia en el Decision Tree:


,Feature,Importance
4,is_monthly_contract,0.514101
1,tenure,0.164449
13,InternetService_Fiber optic,0.163303
2,MonthlyCharges,0.040319
3,TotalCharges,0.039772
6,uses_electronic_check,0.030120
14,InternetService_No,0.016642
28,Contract_Two year,0.012504
22,TechSupport_Yes,0.005448
26,StreamingMovies_Yes,0.003850


### Variables creadas en Feature Engineering

También revisamos si las variables creadas en la fase anterior aparecen entre las variables utilizadas por los modelos.

Nos interesa especialmente `tenure_group`, `monthly_charge_group`, `is_monthly_contract`, `num_services_contracted` y `uses_electronic_check`.


In [13]:
engineered_features = [
    "tenure_group",
    "monthly_charge_group",
    "is_monthly_contract",
    "num_services_contracted",
    "uses_electronic_check"
]

available_engineered = [
    feature for feature in engineered_features
    if feature in X_train.columns
]

print("Features de Feature Engineering disponibles:")
print(available_engineered)


Features de Feature Engineering disponibles:
['tenure_group', 'monthly_charge_group', 'is_monthly_contract', 'num_services_contracted', 'uses_electronic_check']


### Interpretación de los resultados

En la Regresión Logística, las variables con mayor peso absoluto son 
- `Contract_Two year`
- `InternetService_Fiber optic`
- `PhoneService_Yes`
- `is_monthly_contract`
- `PaperlessBilling_Yes`.

En el Decision Tree destaca:
- `is_monthly_contract`
- `tenure`
- `InternetService_Fiber optic`.

Esto indica que el tipo de contrato tiene bastante peso en ambos modelos. También aparecen variables relacionadas con la antigüedad, el tipo de internet y la facturación.


## 12. Análisis de errores

Las métricas anteriores nos dicen qué rendimiento tiene cada modelo, pero no muestran por sí solas qué tipo de errores está cometiendo.

Por eso revisamos los cuatro casos posibles:

- **True Negative:** el cliente no hace churn y el modelo acierta.
- **False Positive:** el modelo predice churn, pero el cliente no abandona.
- **False Negative:** el cliente hace churn, pero el modelo no lo detecta.
- **True Positive:** el cliente hace churn y el modelo lo detecta.

En este proyecto los False Negatives son especialmente importantes porque representan clientes que podrían quedar fuera de una acción de retención.


In [14]:
def error_counts(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred, labels=["No", "Yes"])
    tn, fp, fn, tp = cm.ravel()
    return {
        "True Negative": tn,
        "False Positive": fp,
        "False Negative": fn,
        "True Positive": tp
    }

logistic_errors = error_counts(y_test, y_pred_logistic)
tree_errors = error_counts(y_test, y_pred_tree)

error_comparison = pd.DataFrame(
    [logistic_errors, tree_errors],
    index=["Logistic Regression", "Decision Tree"]
)

display(error_comparison)


,True Negative,False Positive,False Negative,True Positive
Logistic Regression,926,109,172,202
Decision Tree,908,127,156,218


### Interpretación de los errores

Los resultados muestran:.

- Decision Tree tiene **16 False Negatives menos**, por lo que detecta mejor los clientes que realmente hacen churn. También consigue 16 True Positives más.

- La Regresión Logística, por otro lado, genera **18 False Positives menos**, por lo que produce menos clientes marcados como riesgo cuando finalmente no abandonan.

Por tanto, existe un intercambio claro entre detectar más churn y generar más falsas alarmas.


## 13. Selección del modelo final

Para elegir el modelo final no tenemos en cuenta solo el Accuracy. En este proyecto es especialmente importante detectar a los clientes que realmente hacen churn, por lo que damos más importancia al **Recall** y a los **False Negatives**.

In [15]:
comparison = results.copy()
comparison["False Negatives"] = [
    logistic_errors["False Negative"],
    tree_errors["False Negative"]
]
comparison["False Positives"] = [
    logistic_errors["False Positive"],
    tree_errors["False Positive"]
]

display(comparison.round(3))

print(
    "La selección final debe considerar Recall, F1, errores, "
    "interpretabilidad y utilidad de negocio."
)


,Accuracy,Precision,Recall,F1,False Negatives,False Positives
Model,,,,,,
Logistic Regression,0.801,0.650,0.540,0.590,172,109
Decision Tree,0.799,0.632,0.583,0.606,156,127


La selección final debe considerar Recall, F1, errores, interpretabilidad y utilidad de negocio.


### Decisión final
**Decision Tree:** 
- mejor Recall (0.583 frente a 0.540)
- mejor F1 (0.606 frente a 0.590)
- menos False Negatives (156 frente a 172)

**Logistic Regression:** 
- Accuracy ligeramente superior (0.801 frente a 0.799)
- mejor Precision (0.650 frente a 0.632) 
- menos False Positives (109 frente a 127)

Como el objetivo principal es detectar clientes que pueden abandonar, se considera más importante reducir los clientes con churn que el modelo no consigue identificar.

El Decision Tree obtiene mejores resultados en Recall y F1, y además tiene menos False Negatives. Esto es importante porque detectar clientes que realmente abandonan es una prioridad.

Finalmente, se selecciona el **Decision Tree** como modelo final.




## 14. Análisis de False Negatives del modelo seleccionado

Los False Negatives son clientes que realmente hacen churn, pero que el modelo predice como clientes que no abandonarán.

Como el Decision Tree es el modelo seleccionado, vamos a revisar estos casos para entender mejor qué tipo de clientes el modelo no consigue detectar.

In [16]:
selected_model_name = "Decision Tree"

if selected_model_name == "Decision Tree":
    selected_model = tree_model
    selected_predictions = y_pred_tree
else:
    selected_model = logistic_model
    selected_predictions = y_pred_logistic

test_results = X_test.copy()
test_results["Actual_Churn"] = y_test.values
test_results["Predicted_Churn"] = selected_predictions

false_negatives = test_results[
    (test_results["Actual_Churn"] == "Yes") &
    (test_results["Predicted_Churn"] == "No")
]

print("Modelo seleccionado:", selected_model_name)
print("Número de False Negatives:", len(false_negatives))

display(false_negatives.head(10))

Modelo seleccionado: Decision Tree
Número de False Negatives: 156


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,...,PaymentMethod,MonthlyCharges,TotalCharges,tenure_group,monthly_charge_group,is_monthly_contract,num_services_contracted,uses_electronic_check,Actual_Churn,Predicted_Churn
1639,Male,1,No,No,17,Yes,No,DSL,No,No,...,Electronic check,45.05,770.60,13-24 months,low,1,0,1,Yes,No
2488,Male,0,No,No,31,Yes,No,DSL,No,No,...,Electronic check,55.25,1715.65,25-48 months,medium,1,2,1,Yes,No
523,Female,0,No,No,23,Yes,Yes,Fiber optic,No,No,...,Bank transfer (automatic),75.60,1758.60,13-24 months,medium,1,0,0,Yes,No
5086,Male,0,No,No,45,Yes,No,No,No internet service,No internet service,...,Electronic check,20.40,930.45,25-48 months,low,0,0,1,Yes,No
3408,Female,0,No,No,4,Yes,No,DSL,No,Yes,...,Credit card (automatic),50.70,151.30,0-12 months,medium,1,1,0,Yes,No
7011,Female,0,No,No,4,Yes,No,DSL,Yes,Yes,...,Mailed check,60.40,272.15,0-12 months,medium,1,3,0,Yes,No
3158,Male,0,Yes,No,18,Yes,No,DSL,No,No,...,Mailed check,49.55,878.35,13-24 months,low,1,1,0,Yes,No
5968,Female,0,No,No,12,Yes,Yes,DSL,No,No,...,Mailed check,74.05,872.65,0-12 months,medium,0,3,0,Yes,No
1577,Female,0,Yes,Yes,17,Yes,No,Fiber optic,No,No,...,Bank transfer (automatic),98.60,1704.95,13-24 months,high,0,4,0,Yes,No
3975,Female,1,No,No,20,Yes,Yes,Fiber optic,No,No,...,Credit card (automatic),85.80,1727.50,13-24 months,high,1,1,0,Yes,No


In [17]:
print("Características de los False Negatives:")

display(
    false_negatives[
        ["tenure", "MonthlyCharges", "TotalCharges",
         "Contract", "InternetService", "PaymentMethod"]
    ].describe()
)

Características de los False Negatives:


,tenure,MonthlyCharges,TotalCharges
count,156.000000,156.000000,156.000000
mean,28.782051,69.287821,2460.115705
std,21.363307,30.935098,2251.418277
min,1.000000,18.950000,19.300000
25%,9.000000,47.912500,291.887500
50%,26.500000,72.200000,1878.200000
75%,47.000000,98.375000,3927.325000
max,72.000000,115.650000,7968.850000


In [18]:
print("Contract:")
display(false_negatives["Contract"].value_counts())

print("InternetService:")
display(false_negatives["InternetService"].value_counts())

print("PaymentMethod:")
display(false_negatives["PaymentMethod"].value_counts())

Contract:


Contract
Month-to-month    111
One year           36
Two year            9
Name: count, dtype: int64

InternetService:


InternetService
Fiber optic    77
DSL            54
No             25
Name: count, dtype: int64

PaymentMethod:


PaymentMethod
Electronic check             53
Bank transfer (automatic)    37
Credit card (automatic)      34
Mailed check                 32
Name: count, dtype: int64

El modelo obtiene **156 False Negatives**. 

- antigüedad media es aproximadamente **29 meses**
- gasto mensual medio es de unos **69**.

En cuanto al tipo de contrato:
-  mayoría contrato **Month-to-month** (111 casos)
- contratos de un año (36) 
- dos años (9)

Respecto al servicio de Internet:
- **77** fibra óptica
- **54** DSL 
- **25** no tienen servicio de Internet.

En el método de pago: e
- **Electronic check** tiene 53 casos. 
- resto se reparte entre transferencia bancaria automática, tarjeta de crédito automática y cheque.

Estos resultados permiten identificar algunos patrones dentro de los clientes que el modelo no consigue detectar. Sin embargo, este análisis es descriptivo y no permite afirmar que estas características sean la causa del churn o del error del modelo.

Por tanto, los **156 False Negatives** muestran que todavía existe un grupo de clientes con churn que el modelo no consigue identificar correctamente. Esto es importante tenerlo en cuenta si el modelo se utiliza posteriormente para realizar acciones de retención.

## 16. Guardado del modelo

Una vez elegido el modelo, lo guardamos en `../models/churn_model.joblib`.

También guardamos las columnas utilizadas y los valores necesarios para reproducir la preparación de los datos.

Así podemos conservar el modelo y saber cómo se construyeron las variables que utilizó durante el entrenamiento.


In [19]:
import joblib
import os

os.makedirs("../models", exist_ok=True)

model_artifact = {
    "model": selected_model,
    "model_name": selected_model_name,
    "feature_columns": X_train_encoded.columns.tolist(),
    "categorical_columns": categorical_columns,
    "monthly_charge_q1": monthly_charge_q1 if "MonthlyCharges" in X_train.columns else None,
    "monthly_charge_q2": monthly_charge_q2 if "MonthlyCharges" in X_train.columns else None
}

model_path = "../models/churn_model.joblib"
joblib.dump(model_artifact, model_path)

print("Modelo guardado en:", model_path)


Modelo guardado en: ../models/churn_model.joblib
